## Gesture Controls for Air Drawing

- ✴️ **Color / Clear Tab Activation**  
  → Hold your **index finger tip** near a color box or the "Clear" button to activate it.

- 🖌️ **Brush Thickness Control**  
  → The **distance between the thumb tip and index tip** controls brush thickness:  
  Closer = Thin | Farther = Thick

- ❌ **Exit the Program**  
  → Press **`q`** on the keyboard to quit the application.


In [1]:
import cv2
import mediapipe as mp
import numpy as np
import math
import time

In [2]:
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.8, #keeping it high for precision during drawing
    min_tracking_confidence=0.6 #optimal between  missing frame and random flicks
)
mp_drawing = mp.solutions.drawing_utils

In [3]:
canvas_width, canvas_height = 1280, 720
drawing_canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8) #rgb channels is 3 and rgba is 4

In [4]:
current_color = (255, 255, 255)
current_thickness = 5
prev_x, prev_y = 0, 0  #to draw straight lines
is_drawing = False
last_color_change_time = 0
color_cooldown = 1.0  # avoid multiple taps on same button

In [5]:

# color map with box coord  (x,y,w,h) and rgb values
color_palette = {
    "red": {"rect": (50, 10, 60, 40), "color": (0, 0, 255)},
    "green": {"rect": (130, 10, 60, 40), "color": (0, 255, 0)},
    "blue": {"rect": (210, 10, 60, 40), "color": (255, 0, 0)},
    "yellow": {"rect": (290, 10, 60, 40), "color": (0, 255, 255)},
    "cyan": {"rect": (370, 10, 60, 40), "color": (255, 255, 0)},
    "magenta": {"rect": (450, 10, 60, 40), "color": (255, 0, 255)},
    "white": {"rect": (530, 10, 60, 40), "color": (255, 255, 255)},
    "eraser": {"rect": (610, 10, 60, 40), "color": (0, 0, 0)}
}
clear_button_rect = (canvas_width - 150, 10, 120, 50)  # x  measured from right of screen

In [6]:
cap = cv2.VideoCapture(0) #0 is the deafult camera of system
cap.set(cv2.CAP_PROP_FRAME_WIDTH, canvas_width)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, canvas_height)  #resize request to webcams


True

## MediaPipe Hand Landmarks (21 Points)

| Index | Landmark Name        | Description                      |
|-------|----------------------|----------------------------------|
| 0     | WRIST                | Base of the palm                 |
| 1     | THUMB_CMC            | Thumb carpometacarpal joint      |
| 2     | THUMB_MCP            | Thumb metacarpophalangeal joint |
| 3     | THUMB_IP             | Thumb interphalangeal joint      |
| 4     | THUMB_TIP            | Thumb tip                        |
| 5     | INDEX_FINGER_MCP     | Index base joint                 |
| 6     | INDEX_FINGER_PIP     | Index middle joint               |
| 7     | INDEX_FINGER_DIP     | Index top joint                  |
| 8     | INDEX_FINGER_TIP     | Index fingertip                  |
| 9     | MIDDLE_FINGER_MCP    | Middle base joint                |
| 10    | MIDDLE_FINGER_PIP    | Middle middle joint              |
| 11    | MIDDLE_FINGER_DIP    | Middle top joint                 |
| 12    | MIDDLE_FINGER_TIP    | Middle fingertip                 |
| 13    | RING_FINGER_MCP      | Ring base joint                  |
| 14    | RING_FINGER_PIP      | Ring middle joint                |
| 15    | RING_FINGER_DIP      | Ring top joint                   |
| 16    | RING_FINGER_TIP      | Ring fingertip                   |
| 17    | PINKY_MCP            | Pinky base joint                 |
| 18    | PINKY_PIP            | Pinky middle joint               |
| 19    | PINKY_DIP            | Pinky top joint                  |
| 20    | PINKY_TIP            | Pinky fingertip                  |


In [7]:
if not cap.isOpened():
    print("Webcam is not there.")
    exit()

while True:
    ret, frame = cap.read()  # ret is a boolean for succesfl image capture
    if not ret:
        break

    frame = cv2.flip(frame, 1) #mirror
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  #opencv issue :)
    results = hands.process(rgb_frame)   #get 21 2D landmarks per hand ,hands classification L/R results, 3D landmarks

   
    gray_drawing = cv2.cvtColor(drawing_canvas, cv2.COLOR_BGR2GRAY)  #need as threshold require single channel
    _, mask = cv2.threshold(gray_drawing, 1, 255, cv2.THRESH_BINARY) 
    #we could have done frame = frame + drawing_canvas but thats lead to uncontrolled pixel bleed
 
    mask_inv = cv2.bitwise_not(mask)  #white background  zero pixels
    frame_bg = cv2.bitwise_and(frame, frame, mask=mask_inv)  #only background
    drawing_fg = cv2.bitwise_and(drawing_canvas, drawing_canvas, mask=mask) #only brush
    combined_frame = cv2.add(frame_bg, drawing_fg)  #its like doing two operation on diff image and then merge to get single image
    #for the frame u remove parts where the brush will be present and then add the extracted brush strokes from the canvas to here

    for name, data in color_palette.items():
        x, y, w, h = data["rect"]
        cv2.rectangle(combined_frame, (x, y), (x + w, y + h), data["color"], -1)  #filled rect
        cv2.rectangle(combined_frame, (x, y), (x + w, y + h), (200, 200, 200), 2) #border
        text_color = (0, 0, 0) if name != "eraser" else (255, 255, 255) 
        cv2.putText(combined_frame, name.capitalize(), (x + 3, y + h - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)


    cx, cy, cw, ch = clear_button_rect
    cv2.rectangle(combined_frame, (cx, cy), (cx + cw, cy + ch), (50, 50, 200), -1)
    cv2.rectangle(combined_frame, (cx, cy), (cx + cw, cy + ch), (200, 200, 200), 2)
    cv2.putText(combined_frame, "CLEAR", (cx + 20, cy + ch - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # brush size and color on bottom left
    cv2.rectangle(combined_frame, (10, canvas_height - 60), (200, canvas_height - 10), (50, 50, 50), -1)  #top left bottom right
    cv2.putText(combined_frame, "Brush:", (20, canvas_height - 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.circle(combined_frame, (160, canvas_height - 35), 10, current_color, -1)
    cv2.putText(combined_frame, f"{current_thickness}px", (20, canvas_height - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:  #loop through 21 points of each hand if two hand then loop run twice
            mp_drawing.draw_landmarks(combined_frame, hand_landmarks, mp_hands.HAND_CONNECTIONS) #draw the skeleton

            h, w, _ = frame.shape
            index_tip = hand_landmarks.landmark[mp_hands.HandLandmark.INDEX_FINGER_TIP] # normalised b/w 0 and 1
            curr_x, curr_y = int(index_tip.x * w), int(index_tip.y * h)

            def finger_up(tip, mcp): #measured from top   tip and base joint
                return hand_landmarks.landmark[tip].y < hand_landmarks.landmark[mcp].y

            index_up = finger_up(mp_hands.HandLandmark.INDEX_FINGER_TIP,
                                 mp_hands.HandLandmark.INDEX_FINGER_MCP)
            middle_down = not finger_up(mp_hands.HandLandmark.MIDDLE_FINGER_TIP,
                                        mp_hands.HandLandmark.MIDDLE_FINGER_MCP)
            ring_down = not finger_up(mp_hands.HandLandmark.RING_FINGER_TIP,
                                      mp_hands.HandLandmark.RING_FINGER_MCP)
            pinky_down = not finger_up(mp_hands.HandLandmark.PINKY_TIP,
                                       mp_hands.HandLandmark.PINKY_MCP)

            if index_up and middle_down and ring_down and pinky_down: #only index up and rest 3 down
                is_drawing = True
                cv2.putText(combined_frame, "DRAWING", (50, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            else:
                is_drawing = False

            # Brush thickness using thumb distance from index tip
            thumb_tip = hand_landmarks.landmark[mp_hands.HandLandmark.THUMB_TIP]
            thumb_x, thumb_y = int(thumb_tip.x * w), int(thumb_tip.y * h)
            dist = math.hypot(curr_x - thumb_x, curr_y - thumb_y)  #eucledian dist
            # Map the distance between thumb and index finger to brush thickness.
# If the fingers are close (distance ~20), brush is thin (2 px).
# If the fingers are far apart (distance ~150), brush is thick (30 px).
            current_thickness = int(np.interp(dist, [20, 150], [2, 30]))
# This prevents it from going out of bounds due to hand jitter or bad tracking.
            current_thickness = max(2, min(current_thickness, 30))


            # Color selection
            for name, data in color_palette.items():
                x, y, w, h = data["rect"]
                if x < curr_x < x + w and y < curr_y < y + h:  #location of index tip
                    if time.time() - last_color_change_time > color_cooldown:
                        current_color = (0, 0, 0) if name == "eraser" else data["color"]
                        last_color_change_time = time.time()
                        is_drawing = False
                        cv2.rectangle(combined_frame, (x, y), (x + w, y + h), (255, 255, 255), 3)

            if cx < curr_x < cx + cw and cy < curr_y < cy + ch:
                if time.time() - last_color_change_time > color_cooldown:
                    drawing_canvas = np.zeros((canvas_height, canvas_width, 3), dtype=np.uint8)
                    last_color_change_time = time.time()
                    is_drawing = False
                    cv2.rectangle(combined_frame, (cx, cy), (cx + cw, cy + ch), (255, 255, 255), 3)

            #draw line 
            if is_drawing:
                if prev_x == 0 and prev_y == 0:
                    prev_x, prev_y = curr_x, curr_y
                cv2.line(drawing_canvas, (prev_x, prev_y), (curr_x, curr_y), current_color, current_thickness)
                prev_x, prev_y = curr_x, curr_y
            else:
                prev_x, prev_y = 0, 0

    else:
        is_drawing = False
        prev_x, prev_y = 0, 0

    cv2.imshow('Drawing Canvas', combined_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):  #ascii of  q
        break

In [8]:
cap.release() #close webcam stream
cv2.destroyAllWindows()